# Notebook 02 — Parameter Estimation

| Field | Detail |
|---|---|
| **Member** | Member B — Estimation Analyst |
| **Project** | `moby/moby` (Docker Engine) |
| **Layer** | Week 11 — MLE + Beta Posterior |

## Research Questions Addressed

> **RQ1:** *What is the estimated probability that a Pull Request in `moby/moby` gets merged, and how does the Beta posterior distribution characterise our uncertainty around that estimate?*
>
> **RQ2:** *What is the estimated average rate of new issues opened per month in `moby/moby`, and does this rate follow a Poisson process?*

---
**Upstream dependency:** This notebook consumes `data/clean/pull_requests_clean.csv`, `issues_clean.csv`, and `commits_clean.csv` prepared by **Member A (Data Engineer)**.

**Downstream dependency:** The point estimates (`theta_hat`, `lambda_hat`, `alpha`, `beta`) produced here are consumed by **Member C (Inference Analyst)** in `03_confidence_interval.ipynb`.

## AI Disclosure

| # | Task | Tool | Prompt Summary | How output was used |
|---|------|------|---------------|---------------------|
| 1 | Initial `estimator.py` scaffold | Claude | "Generate MLE functions for Bernoulli and Poisson per Tsun 2020" | Reviewed, corrected α/β formula to k+1/m+1, kept structure |
| 2 | Notebook markdown cells | Claude | "Write interpretation for Beta posterior result" | Rewritten in own words; formula checks done manually |

**Written entirely without AI:** All statistical interpretation paragraphs, RQ framing, and summary conclusions.

---
## 0. Environment Setup

In [ ]:
import sys
import os

# Make sure src/ is on the path — notebooks run from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# If running from project root:
# sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import beta as beta_dist

# Import our own module (Member B)
from estimator import (
    mle_bernoulli,
    mle_poisson,
    beta_posterior,
    log_likelihood_bernoulli,
    log_likelihood_poisson,
    plot_log_likelihood_bernoulli,
    plot_log_likelihood_poisson,
    plot_beta_posterior,
)

# Plot styling
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('muted')

print('Environment ready ✓')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---
## 1. Load & Inspect Data

In [ ]:
# Paths — adjust if running from a different working directory
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'clean')

pr_df     = pd.read_csv(os.path.join(DATA_DIR, 'pull_requests_clean.csv'), parse_dates=['created_at', 'merged_at', 'closed_at'])
issues_df = pd.read_csv(os.path.join(DATA_DIR, 'issues_clean.csv'),        parse_dates=['created_at', 'closed_at'])
commits_df= pd.read_csv(os.path.join(DATA_DIR, 'commits_clean.csv'),       parse_dates=['author_date'])

print(f'PRs     : {len(pr_df):,} rows | cols: {list(pr_df.columns)}')
print(f'Issues  : {len(issues_df):,} rows | cols: {list(issues_df.columns)}')
print(f'Commits : {len(commits_df):,} rows | cols: {list(commits_df.columns)}')

In [ ]:
# Quick sanity check on PR merge column
print('PR is_merged value counts:')
print(pr_df['is_merged'].value_counts())

print('\nPR state value counts:')
print(pr_df['state'].value_counts())

---
## 2. RQ1 — MLE Bernoulli: Probability a PR Gets Merged

### 2.1 Motivation

Every PR in `moby/moby` has a binary outcome: **merged** (success) or **not merged** (closed/open). We model this as a Bernoulli random variable:

$$X_i \sim \text{Bernoulli}(\theta), \quad X_i \in \{0, 1\}$$

The goal is to find the value of θ that maximises the likelihood of observing our data.

### 2.2 MLE Derivation (Tsun, 2020, p. 254)

Given n independent Bernoulli trials with k successes:

$$L(\theta) = \theta^k (1-\theta)^{n-k}$$

Taking the log and differentiating with respect to θ:

$$\ell(\theta) = k \ln\theta + (n-k)\ln(1-\theta)$$

$$\frac{d\ell}{d\theta} = \frac{k}{\theta} - \frac{n-k}{1-\theta} = 0 \implies \boxed{\hat{\theta} = \frac{k}{n}}$$

In [ ]:
# --- Prepare binary outcome column ---
# We use all PRs that have a definitive outcome (merged or closed, not still open)
pr_closed = pr_df[pr_df['state'].isin(['closed'])].copy()
pr_closed['outcome'] = pr_closed['is_merged'].astype(int)

print(f'Closed PRs (definitive outcome): {len(pr_closed):,}')
print(f'  Merged : {pr_closed["outcome"].sum():,}')
print(f'  Closed without merge: {(pr_closed["outcome"] == 0).sum():,}')

In [ ]:
# --- Compute MLE Bernoulli using estimator.py ---
bern_result = mle_bernoulli(pr_closed['outcome'].tolist())

print('=== MLE Bernoulli Results ===')
print(f"  θ̂ (merge probability)  : {bern_result['theta_hat']:.4f}")
print(f"  k (merged PRs)         : {bern_result['k']:,}")
print(f"  n (total closed PRs)   : {bern_result['n']:,}")
print(f"  Standard error SE(θ̂)  : {bern_result['std_error']:.4f}")

### 2.3 Interpretation

The MLE estimate tells us that approximately **`theta_hat * 100`%** of Pull Requests in `moby/moby` are successfully merged. This is the single value of θ most consistent with the observed data. The standard error of the estimate is small, indicating high precision given the large sample size.

### 2.4 Log-Likelihood Curve

In [ ]:
fig = plot_log_likelihood_bernoulli(
    k=bern_result['k'],
    n=bern_result['n'],
    save_path='../report/fig01_ll_bernoulli.png'
)
plt.show()
print('Figure saved → report/fig01_ll_bernoulli.png')

**Interpretation of the log-likelihood curve:**

The curve has a single global maximum at θ̂ = k/n, which is the MLE. The shape of the curve around the peak also gives us a visual sense of estimator uncertainty — the sharper the peak (more curvature), the lower the variance. With a large dataset like `moby/moby`, the peak is steep, confirming a low standard error.

---
## 3. RQ1 (Bayesian) — Beta Posterior for Merge Probability

### 3.1 Motivation

Instead of a single-point MLE, we can compute the **full posterior distribution** over θ using a conjugate Beta prior. This gives us a richer picture of uncertainty.

### 3.2 Formula (Tsun, 2020, p. 269)

Starting with a **uniform prior** Beta(1, 1), after observing k successes and m failures:

$$p(\theta | k, m) = \text{Beta}(\alpha, \beta), \quad \alpha = k+1, \quad \beta = m+1$$

- **Posterior mode:** $\hat{\theta}_{\text{mode}} = \dfrac{\alpha - 1}{\alpha + \beta - 2}$
- **Posterior mean:** $\hat{\theta}_{\text{mean}} = \dfrac{\alpha}{\alpha + \beta}$

> ⚠️ **Critical note:** α = k+1 and β = m+1, **never** k and m directly. Using bare k/m is a formula error (Tsun, 2020, p. 269).

In [ ]:
k_merges  = bern_result['k']
m_rejects = bern_result['n'] - bern_result['k']   # failures = n - k

bp = beta_posterior(k=k_merges, m=m_rejects)

print('=== Beta Posterior Results ===')
print(f"  α (alpha) = k+1      : {bp['alpha']:,}")
print(f"  β (beta)  = m+1      : {bp['beta']:,}")
print(f"  Posterior mode       : {bp['mode']:.4f}")
print(f"  Posterior mean       : {bp['mean']:.4f}")
print(f"  MLE θ̂ (for comparison): {bern_result['theta_hat']:.4f}")

print('\nNote: mode ≈ MLE θ̂ (they converge with large n, Tsun p.269)')

In [ ]:
fig = plot_beta_posterior(
    k=k_merges,
    m=m_rejects,
    save_path='../report/fig02_beta_posterior.png'
)
plt.show()
print('Figure saved → report/fig02_beta_posterior.png')

### 3.3 Interpretation

The Beta posterior distribution shows the **full range of plausible values** for θ, weighted by the evidence from our data. Key observations:

- The distribution is **highly concentrated** near the MLE, reflecting the large sample size (n > 1,000 PRs). With so much data, the posterior is almost entirely driven by the likelihood, not the prior.
- The **posterior mode** equals the MLE (k/n) up to a negligible correction of 1/(n+2), confirming that both the frequentist and Bayesian point estimates agree at this sample size.
- The **posterior mean** is slightly pulled toward 0.5 compared to the mode — this is the shrinkage effect of the uniform prior Beta(1,1).

This Beta posterior will be handed off to **Member C** for computing the **95% Bayesian credible interval** (the equal-tailed interval covering 95% of the posterior mass).

---
## 4. RQ2 — MLE Poisson: Issue Opening Rate

### 4.1 Motivation

The number of new issues opened per month in `moby/moby` is a **count process** — a natural candidate for Poisson modelling. We want to estimate the underlying average rate λ (issues/month).

### 4.2 MLE Derivation (Tsun, 2020, p. 254)

For n independent Poisson observations $x_1, \ldots, x_n$:

$$L(\lambda) = \prod_{i=1}^{n} \frac{\lambda^{x_i} e^{-\lambda}}{x_i!}$$

Log-likelihood (dropping constant term):

$$\ell(\lambda) = \left(\sum x_i\right)\ln\lambda - n\lambda$$

$$\frac{d\ell}{d\lambda} = \frac{\sum x_i}{\lambda} - n = 0 \implies \boxed{\hat{\lambda} = \frac{\sum x_i}{n} = \bar{x}}$$

In [ ]:
# --- Issues per month count ---
issues_df['month'] = issues_df['created_at'].dt.to_period('M')
issues_per_month = issues_df.groupby('month').size().reset_index(name='count')

print('Issues per month (sample):')
print(issues_per_month.tail(10).to_string(index=False))
print(f'\nTotal months with data: {len(issues_per_month)}')

In [ ]:
# --- Compute MLE Poisson using estimator.py ---
count_data = issues_per_month['count'].tolist()
pois_result = mle_poisson(count_data)

print('=== MLE Poisson Results ===')
print(f"  λ̂ (avg issues/month) : {pois_result['lambda_hat']:.4f}")
print(f"  n (months observed)  : {pois_result['n']}")
print(f"  Standard error       : {pois_result['std_error']:.4f}")

### 4.3 Visualisation — Distribution of Monthly Issue Counts

In [ ]:
from scipy.stats import poisson

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: histogram of monthly counts
ax = axes[0]
ax.hist(count_data, bins=15, color='#059669', edgecolor='white', alpha=0.85)
ax.axvline(pois_result['lambda_hat'], color='#dc2626', linestyle='--', linewidth=2,
           label=f"λ̂ = {pois_result['lambda_hat']:.2f}")
ax.set_xlabel('Issues Opened per Month', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Distribution of Monthly Issue Counts', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)

# Right: log-likelihood curve
lambda_hat = pois_result['lambda_hat']
lo = max(0.1, lambda_hat * 0.4)
hi = lambda_hat * 2.0
lam_range = np.linspace(lo, hi, 500)
ll_vals = log_likelihood_poisson(lam_range, np.array(count_data))

ax2 = axes[1]
ax2.plot(lam_range, ll_vals, color='#059669', linewidth=2.5)
ax2.axvline(lambda_hat, color='#dc2626', linestyle='--', linewidth=1.8,
            label=f'MLE λ̂ = {lambda_hat:.2f}')
ax2.scatter([lambda_hat], [log_likelihood_poisson(lambda_hat, np.array(count_data))],
            color='#dc2626', s=80, zorder=5)
ax2.set_xlabel('λ (Poisson Rate)', fontsize=12)
ax2.set_ylabel('ℓ(λ | x)', fontsize=12)
ax2.set_title('Poisson Log-Likelihood Curve', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.25)

fig.suptitle('moby/moby — Issue Opening Rate (Poisson MLE)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('../report/fig03_poisson_issues.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → report/fig03_poisson_issues.png')

### 4.4 Interpretation

The MLE estimate λ̂ represents the **average number of new issues opened per month** in `moby/moby`. The log-likelihood curve confirms that this is the unique maximum — the Poisson MLE is always the sample mean.

Examining the histogram, we can observe whether the data reasonably follows a Poisson shape (roughly symmetric around the mean for large λ, skewed-right for small λ). Any notable deviations — such as bimodality or overdispersion — should be flagged for **Member D (Hypothesis Analyst)** to investigate in a hypothesis test.

---
## 5. Bonus — Commit Activity Rate (Poisson MLE)

In [ ]:
# Commits per week as a Poisson process
commits_df['week'] = commits_df['author_date'].dt.to_period('W')
commits_per_week = commits_df.groupby('week').size().reset_index(name='count')

commit_result = mle_poisson(commits_per_week['count'].tolist())

print('=== MLE Poisson — Commits per Week ===')
print(f"  λ̂ (avg commits/week) : {commit_result['lambda_hat']:.4f}")
print(f"  n (weeks observed)   : {commit_result['n']}")
print(f"  Standard error       : {commit_result['std_error']:.4f}")

---
## 6. Summary of Estimation Results

| Parameter | Estimator | Formula | Value | SE |
|---|---|---|---|---|
| θ (PR merge prob.) | MLE Bernoulli | k/n | see above | see above |
| α (Beta posterior) | Conjugate update | k+1 | see above | — |
| β (Beta posterior) | Conjugate update | m+1 | see above | — |
| Posterior mode | Beta property | (α−1)/(α+β−2) | see above | — |
| Posterior mean | Beta property | α/(α+β) | see above | — |
| λ (issues/month) | MLE Poisson | Σxᵢ/n | see above | see above |
| λ (commits/week) | MLE Poisson | Σxᵢ/n | see above | see above |

In [ ]:
# Export key estimates to a CSV for Member C
import json

estimates = {
    'bernoulli_theta_hat': bern_result['theta_hat'],
    'bernoulli_k':          bern_result['k'],
    'bernoulli_n':          bern_result['n'],
    'bernoulli_se':         bern_result['std_error'],
    'beta_alpha':           bp['alpha'],
    'beta_beta':            bp['beta'],
    'beta_mode':            bp['mode'],
    'beta_mean':            bp['mean'],
    'poisson_lambda_issues': pois_result['lambda_hat'],
    'poisson_se_issues':     pois_result['std_error'],
    'poisson_n_issues':      pois_result['n'],
    'poisson_lambda_commits': commit_result['lambda_hat'],
    'poisson_n_commits':      commit_result['n'],
}

pd.Series(estimates).to_csv('../data/clean/estimation_results.csv', header=['value'])

print('Estimates exported → data/clean/estimation_results.csv')
print('\nAll values for Member C (Inference Analyst):')
for k, v in estimates.items():
    print(f'  {k:40s}: {v}')

---
## 7. Link to Next Layer

The following outputs from this notebook are consumed by **Member C — Inference Analyst** in `03_confidence_interval.ipynb`:

| Variable | Value | Purpose in Layer C |
|---|---|---|
| `theta_hat` | Bernoulli MLE | Centre of the frequentist CI for merge rate |
| `std_error` | SE(θ̂) | Used in CI formula: θ̂ ± z·SE |
| `n` | Sample size | Required for CI formula |
| `alpha`, `beta` | Beta parameters | Used for credible interval via `beta_dist.ppf()` |
| `lambda_hat` | Poisson MLE | Centre of Poisson CI |

All values are saved to `data/clean/estimation_results.csv`. Member C should import from `src/inference.py` and load this file — **not** re-implement these estimators inline.

---
*End of Notebook 02 — Estimation Layer*